In [1]:
# Import the required Python libraries.

import torch
import numpy as np
import evaluate

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

# Use GPU if available, otherwise use CPU.
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

# Clear GPU cache if CUDA is available.
if device == "cuda":
    torch.cuda.empty_cache()

Device: cuda


In [2]:
# Define the model checkpoint.
# This model is suitable for SAMSum-style dialogue summarization.

model_ckpt = "philschmid/distilbart-cnn-12-6-samsum"

# Load the tokenizer and model.
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

print("Model loaded:", model_ckpt)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/359 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model loaded: philschmid/distilbart-cnn-12-6-samsum


In [3]:
# Load the SAMSum dataset.
# The dataset contains train, validation, and test splits.

dataset_samsum = load_dataset("knkarthick/samsum")

dataset_samsum

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})

In [4]:
# Print one example to understand the dataset structure.

sample = dataset_samsum["train"][0]

print("Dialogue:")
print(sample["dialogue"])

print("\nSummary:")
print(sample["summary"])

Dialogue:
Amanda: I baked  cookies. Do you want some?
Jerry: Sure!
Amanda: I'll bring you tomorrow :-)

Summary:
Amanda baked cookies and will bring Jerry some tomorrow.


In [6]:
# Splitting the Dataset.

small_train_dataset = dataset_samsum["train"].shuffle(seed=42)
small_val_dataset = dataset_samsum["validation"].shuffle(seed=42)
small_test_dataset = dataset_samsum["test"]

print("Train size:", len(small_train_dataset))
print("Validation size:", len(small_val_dataset))
print("Test size:", len(small_test_dataset))

Train size: 14731
Validation size: 818
Test size: 819


## Preprocessing

In this step, we tokenize:
- `dialogue` as the model input.
- `summary` as the target labels.

In [7]:
# Define maximum input and target lengths.

max_input_length = 512
max_target_length = 96

def convert_examples_to_features(example_batch):
    # Tokenize the dialogue inputs.
    model_inputs = tokenizer(
        example_batch["dialogue"],
        max_length=max_input_length,
        truncation=True
    )

    # Tokenize the target summaries.
    labels = tokenizer(
        text_target=example_batch["summary"],
        max_length=max_target_length,
        truncation=True
    )

    # Add labels to the model inputs.
    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [8]:
# Apply the preprocessing function to train, validation, and test datasets.
# remove_columns removes original text columns after tokenization.

tokenized_train = small_train_dataset.map(
    convert_examples_to_features,
    batched=True,
    remove_columns=small_train_dataset.column_names
)

tokenized_val = small_val_dataset.map(
    convert_examples_to_features,
    batched=True,
    remove_columns=small_val_dataset.column_names
)

tokenized_test = small_test_dataset.map(
    convert_examples_to_features,
    batched=True,
    remove_columns=small_test_dataset.column_names
)

print(tokenized_train)

Map:   0%|          | 0/14731 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 14731
})


In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

In [10]:
# Load the ROUGE metric for summarization evaluation.

rouge_metric = evaluate.load("rouge")

In [11]:
# Define a function to compute ROUGE scores during evaluation.

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # Decode generated summaries.
    decoded_preds = tokenizer.batch_decode(
        predictions,
        skip_special_tokens=True
    )

    # Replace -100 values with pad_token_id before decoding labels.
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    # Decode reference summaries.
    decoded_labels = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True
    )

    # Basic text cleanup.
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]

    # Compute ROUGE.
    result = rouge_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )

    # Round the scores for cleaner output.
    return {key: round(value, 4) for key, value in result.items()}

In [12]:
# Training Arguments

training_args = Seq2SeqTrainingArguments(
    output_dir="distilbart-samsum-colab",

    num_train_epochs=1,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=2e-5,
    weight_decay=0.01,

    predict_with_generate=True,
    generation_max_length=96,
    generation_num_beams=4,

    logging_steps=50,

    eval_strategy="steps",
    eval_steps=500,

    save_strategy="steps",
    save_steps=500,
    save_total_limit=1,

    fp16=torch.cuda.is_available(),

    report_to="none"
)

In [15]:
# Create the Seq2SeqTrainer.

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

In [16]:
# Start training.

trainer.train()

Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
500,9.947236,1.402575,0.412200,0.209100,0.317800,0.317800
1000,9.846324,1.381534,0.420200,0.216000,0.324400,0.324400
1500,9.401217,1.351025,0.419400,0.218800,0.326700,0.326700


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1842, training_loss=9.862234888066428, metrics={'train_runtime': 3840.5341, 'train_samples_per_second': 3.836, 'train_steps_per_second': 0.48, 'total_flos': 3989492351004672.0, 'train_loss': 9.862234888066428, 'epoch': 1.0})

In [17]:
# Evaluate the model on the test set.

test_results = trainer.evaluate(eval_dataset=tokenized_test)

test_results

{'eval_loss': 1.3910155296325684,
 'eval_rouge1': 0.4117,
 'eval_rouge2': 0.2045,
 'eval_rougeL': 0.3134,
 'eval_rougeLsum': 0.3135,
 'eval_runtime': 651.8413,
 'eval_samples_per_second': 1.256,
 'eval_steps_per_second': 1.256,
 'epoch': 1.0}

In [18]:
# Generate a summary for one test example.

idx = 0

dialogue = dataset_samsum["test"][idx]["dialogue"]
reference_summary = dataset_samsum["test"][idx]["summary"]

inputs = tokenizer(
    dialogue,
    return_tensors="pt",
    truncation=True,
    max_length=max_input_length
).to(device)

model.eval()

with torch.no_grad():
    summary_ids = model.generate(
        **inputs,
        max_length=96,
        num_beams=4,
        length_penalty=0.8
    )

generated_summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print("Dialogue:")
print(dialogue)

print("\nReference Summary:")
print(reference_summary)

print("\nGenerated Summary:")
print(generated_summary)

Dialogue:
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye

Reference Summary:
Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.

Generated Summary:
Amanda can't find Betty's number. Larry called her last time they were at the park together. Hannah would rather Amanda to text him.    i will text Larry.  i am not sure if he is a good friend of Hannah or not. 


In [19]:
# Test the model on a custom dialogue.

custom_dialogue = """
Ali: Are you coming to the meeting today?
Sara: Yes, but I might be 10 minutes late.
Ali: No problem, I will update the team.
Sara: Thanks, please tell them I prepared the report.
"""

inputs = tokenizer(
    custom_dialogue,
    return_tensors="pt",
    truncation=True,
    max_length=max_input_length
).to(device)

model.eval()

with torch.no_grad():
    summary_ids = model.generate(
        **inputs,
        max_length=96,
        num_beams=4,
        length_penalty=0.8
    )

summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print(summary)

Sara will be 10 minutes late to the meeting today. Ali will update the team. Sara prepared the report and she will tell them she will be late. .    iReport is due later today and Ali will inform the team about Sara's report.
